# 01 Data Audit

Performs initial audit of:
* schema, data types, duplicate rows, identifier uniqueness
* missing values, patterns
* time coverage, transaction counts by time period
* labelled class balance, fraud rates

#### Summary of Findings:

##### Sparkov:
Train
* Dataset is complete, with no NA values or duplicated rows
* Identifier column: `Unnamed: 0`, `trans_num`
* Timestamp columns: `trans_date_trans_time`, `unix_time`
* Time coverage: `2019-01-01 00:00:18` to `2020-06-21 12:13:37` (537 unique days)
* Number of Daily Transactions: Highest Recorded (`6,428`), Lowest Recorded (`1,073`)
* Class Imbalance: **very high** (99.4% Licit vs 0.5% Illicit)

Test
* Dataset is complete, with no NA values or duplicated rows
* Identifier column: `Unnamed: 0`, `trans_num`
* Timestamp columns: `trans_date_trans_time`, `unix_time`
* Time coverage: `2020-06-21 12:14:25` to `2020-12-31 23:59:34`
* Class Imbalance: **very high** (99.6% Licit vs 0.3% Illicit)

##### IEEE-CIS:
Transaction
* There are no duplicate rows
* Dataset has many NA values for several columns like `D(number)` and `V(number)`
* Other NA columns: `dist2`, `R_emaildomain`, `P_emaildomain`, `addr1`, `addr2`, `card(2-6)`
* Identifier column: `TransactionID`
* Timestamp columns: Not apparent, must calculate from `TransactionID`, elapsed time from reference point

Identity
* There are no duplicate rows
* With the exception of columns `TransactionID`, `id_01`, `id_02`, the rest of the columns have NA values
* Columns `id_07`, `id_08`, `id_21`, `id_22`, `id_23`, `id_24`, `id_25`, `id_26`, `id_27` have `>96%` of NA rows
* Identifier column: `TransactionID`
* With a total of 144,233 `Identity` rows, only 24.42% of `Transaction` rows have matching identity records
* All Identity rows have a matching Transaction


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [8]:

DATA_DIR = Path("../data/raw")

SPARKOV_TRAIN_PATH = DATA_DIR / "fraudTrain.csv"
SPARKOV_TEST_PATH = DATA_DIR / "fraudTest.csv" 

IEEE_TRANSACTION_PATH = DATA_DIR / "train_transaction.csv"
IEEE_IDENTITY_PATH = DATA_DIR / "train_identity.csv"

# Set to False if you want a quicker first run on a large machine.
LOAD_OPTIONAL_FILES = True

In [4]:
def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"File not found: {path.resolve()}\n"
            "Update the configuration cell with the correct CSV path."
        )
    return path

def read_csv_checked(path, **kwargs):
    path = require_file(path)
    print(f"Reading {path} ...")
    return pd.read_csv(path, low_memory=False, **kwargs)

def schema_audit(df, name):
    result = pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "non_null": df.notna().sum().values,
        "missing": df.isna().sum().values,
        "missing_pct": (df.isna().mean().mul(100)).values,
        "n_unique": df.nunique(dropna=True).values,
    }).sort_values(["missing_pct", "column"], ascending=[False, True])
    print(f"{name}: {len(df):,} rows × {df.shape[1]:,} columns")
    print(f"Duplicate rows: {df.duplicated().sum():,}")
    display(result.reset_index(drop=True))
    return result

def time_summary(df, time_col, name, datetime_format=None):
    if time_col not in df.columns:
        print(f"{name}: {time_col!r} not found; time summary skipped.")
        return None
    values = pd.to_datetime(df[time_col], errors="coerce", format=datetime_format)
    valid = values.dropna()
    summary = pd.Series({
        "time_column": time_col,
        "valid_values": len(valid),
        "missing_or_invalid": values.isna().sum(),
        "start": valid.min() if len(valid) else pd.NaT,
        "end": valid.max() if len(valid) else pd.NaT,
        "unique_days": valid.dt.normalize().nunique() if len(valid) else 0,
    }, name=name)
    display(summary.to_frame().T)
    return values

def class_balance(df, label_col, name):
    if label_col not in df.columns:
        print(f"{name}: {label_col!r} not found; class balance skipped.")
        return None
    counts = df[label_col].value_counts(dropna=False).rename_axis(label_col).to_frame("count")
    counts["percentage"] = counts["count"] / len(df) * 100
    display(counts)
    print(f"Labelled rows: {df[label_col].notna().sum():,}; unlabelled rows: {df[label_col].isna().sum():,}")
    return counts

## 1. Sparkov dataset

In [9]:
sparkov_train = read_csv_checked(SPARKOV_TRAIN_PATH)
display(sparkov_train.head())
sparkov_schema = schema_audit(sparkov_train, "Sparkov train")

Reading ..\data\raw\fraudTrain.csv ...


,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.9700,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,NC,28654,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.0113,-82.0483,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.2300,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,WA,99160,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.1590,-118.1865,0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.1100,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,ID,83252,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.1507,-112.1545,0
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.0000,Jeremy,White,M,9443 Cynthia Court Apt. 038,Boulder,MT,59632,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.0343,-112.5611,0
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.9600,Tyler,Garcia,M,408 Bradley Rest,Doe Hill,VA,24433,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.6750,-78.6325,0


Sparkov train: 1,296,675 rows × 23 columns
Duplicate rows: 0


,column,dtype,non_null,missing,missing_pct,n_unique
0,Unnamed: 0,int64,1296675,0,0.0000,1296675
1,amt,float64,1296675,0,0.0000,52928
2,category,object,1296675,0,0.0000,14
3,cc_num,int64,1296675,0,0.0000,983
4,city,object,1296675,0,0.0000,894
5,city_pop,int64,1296675,0,0.0000,879
6,dob,object,1296675,0,0.0000,968
7,first,object,1296675,0,0.0000,352
8,gender,object,1296675,0,0.0000,2
9,is_fraud,int64,1296675,0,0.0000,2


The Sparkov Train dataset has no NA values, no duplicated rows

In [10]:
sparkov_time = time_summary(sparkov_train, "trans_date_trans_time", "Sparkov train")
sparkov_balance = class_balance(sparkov_train, "is_fraud", "Sparkov train")

if "trans_date_trans_time" in sparkov_train.columns:
    sparkov_train["audit_date"] = pd.to_datetime(
        sparkov_train["trans_date_trans_time"], errors="coerce"
    ).dt.date
    display(
        sparkov_train.groupby("audit_date", dropna=False)
        .size().rename("transactions").to_frame().head(10)
    )

,time_column,valid_values,missing_or_invalid,start,end,unique_days
Sparkov train,trans_date_trans_time,1296675,0,2019-01-01 00:00:18,2020-06-21 12:13:37,537


,count,percentage
is_fraud,,
0,1289169,99.4211
1,7506,0.5789


Labelled rows: 1,296,675; unlabelled rows: 0


,transactions
audit_date,
2019-01-01,2414
2019-01-02,1118
2019-01-03,1236
2019-01-04,1474
2019-01-05,1409
2019-01-06,2029
2019-01-07,2425
2019-01-08,2314
2019-01-09,1158


In [23]:
display(sparkov_train.groupby("audit_date", dropna=False).size().min())
display(sparkov_train.groupby("audit_date", dropna=False).size().max())

np.int64(1073)

np.int64(6428)

In [11]:
# Sparkov test-file audit - also consists of labels
if LOAD_OPTIONAL_FILES and Path(SPARKOV_TEST_PATH).exists():
    sparkov_test = read_csv_checked(SPARKOV_TEST_PATH)
    sparkov_test_schema = schema_audit(sparkov_test, "Sparkov test")
    time_summary(sparkov_test, "trans_date_trans_time", "Sparkov test")
    class_balance(sparkov_test, "is_fraud", "Sparkov test")
else:
    print("Optional Sparkov test file not found or disabled.")

Reading ..\data\raw\fraudTest.csv ...
Sparkov test: 555,719 rows × 23 columns
Duplicate rows: 0


,column,dtype,non_null,missing,missing_pct,n_unique
0,Unnamed: 0,int64,555719,0,0.0000,555719
1,amt,float64,555719,0,0.0000,37256
2,category,object,555719,0,0.0000,14
3,cc_num,int64,555719,0,0.0000,924
4,city,object,555719,0,0.0000,849
5,city_pop,int64,555719,0,0.0000,835
6,dob,object,555719,0,0.0000,910
7,first,object,555719,0,0.0000,341
8,gender,object,555719,0,0.0000,2
9,is_fraud,int64,555719,0,0.0000,2


,time_column,valid_values,missing_or_invalid,start,end,unique_days
Sparkov test,trans_date_trans_time,555719,0,2020-06-21 12:14:25,2020-12-31 23:59:34,194


,count,percentage
is_fraud,,
0,553574,99.6140
1,2145,0.3860


Labelled rows: 555,719; unlabelled rows: 0


## 2. IEEE-CIS dataset

In [12]:
ieee_transaction = read_csv_checked(IEEE_TRANSACTION_PATH)
display(ieee_transaction.head())
ieee_transaction_schema = schema_audit(ieee_transaction, "IEEE-CIS train transaction")
ieee_balance = class_balance(ieee_transaction, "isFraud", "IEEE-CIS train transaction")

Reading ..\data\raw\train_transaction.csv ...


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,addr2,dist1,dist2,P_emaildomain,R_emaildomain,C1,C2,C3,C4,C5,C6,C7,C8,C9,C10,C11,C12,C13,C14,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10,D11,D12,D13,D14,D15,M1,M2,M3,M4,M5,M6,M7,M8,M9,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,V29,V30,V31,V32,V33,V34,V35,V36,V37,V38,V39,V40,V41,V42,V43,V44,V45,...,V240,V241,V242,V243,V244,V245,V246,V247,V248,V249,V250,V251,V252,V253,V254,V255,V256,V257,V258,V259,V260,V261,V262,V263,V264,V265,V266,V267,V268,V269,V270,V271,V272,V273,V274,V275,V276,V277,V278,V279,V280,V281,V282,V283,V284,V285,V286,V287,V288,V289,V290,V291,V292,V293,V294,V295,V296,V297,V298,V299,V300,V301,V302,V303,V304,V305,V306,V307,V308,V309,V310,V311,V312,V313,V314,V315,V316,V317,V318,V319,V320,V321,V322,V323,V324,V325,V326,V327,V328,V329,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987000,0,86400,68.5000,W,13926,NaN,150.0000,discover,142.0000,credit,315.0000,87.0000,19.0000,NaN,NaN,NaN,1.0000,1.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,1.0000,0.0000,2.0000,0.0000,1.0000,1.0000,14.0000,NaN,13.0000,NaN,NaN,NaN,NaN,NaN,NaN,13.0000,13.0000,NaN,NaN,NaN,0.0000,T,T,T,M2,F,T,NaN,NaN,NaN,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.0000,0.0000,1.0000,1.0000,1.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000,0.0000,0.0000,1.0000,1.0000,1.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000,0.0000,0.0000,1.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000,1.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,117.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,117.0000,0.0000,0.0000,0.0000,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0000,W,2755,404.0000,150.0000,mastercard,102.0000,credit,325.0000,87.0000,NaN,NaN,gmail.com,NaN,1.0000,1.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,1.0000,1.0000,0.0000,NaN,NaN,0.0000,NaN,NaN,NaN,NaN,NaN,0.0000,NaN,NaN,NaN,NaN,0.0000,NaN,NaN,NaN,M0,T,T,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000,0.0000,0.0000,1.0000,1.0000,1.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000,0.0000,0.0000,1.0000,0.0000,0.0000,1.0000,1.0000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000,0.0000,0.0000,1.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0000,W,4663,490.0000,150.0000,visa,166.0000,debit,330.0000,87.0000,287.0000,NaN,outlook.com,NaN,1.0000,1.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,1.0000,0.0000,1.0000,0.0000,1.0000,1.0000,0.0000,NaN,NaN,0.0000,NaN,NaN,NaN,NaN,NaN,0.0000,315.0000,NaN,NaN,NaN,315.0000,T,T,T,M0,F,F,F,F,F,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.0000,0.0000,1.0000,1.0000,1.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000,0.0000,0.0000,1.0000,1.0000,1.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000,1.0000,1.0000,0.0000,0.0000,1.0000,0.0000,0.0000,1.0000,1.0000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN

IEEE-CIS train transaction: 590,540 rows × 394 columns
Duplicate rows: 0


,column,dtype,non_null,missing,missing_pct,n_unique
0,dist2,float64,37627,552913,93.6284,1751
1,D7,float64,38917,551623,93.4099,597
2,D13,float64,61952,528588,89.5093,577
3,D14,float64,62187,528353,89.4695,802
4,D12,float64,64717,525823,89.0410,635
...,...,...,...,...,...,...
389,TransactionAmt,float64,590540,0,0.0000,20902
390,TransactionDT,int64,590540,0,0.0000,573349
391,TransactionID,int64,590540,0,0.0000,590540
392,card1,int64,590540,0,0.0000,13553


,count,percentage
isFraud,,
0,569877,96.5010
1,20663,3.4990


Labelled rows: 590,540; unlabelled rows: 0


In [20]:
t = ieee_transaction_schema[ieee_transaction_schema["missing"] > 0]

display(t)

print(t["column"].unique())


,column,dtype,non_null,missing,missing_pct,n_unique
14,dist2,float64,37627,552913,93.6284,1751
37,D7,float64,38917,551623,93.4099,597
43,D13,float64,61952,528588,89.5093,577
44,D14,float64,62187,528353,89.4695,802
42,D12,float64,64717,525823,89.0410,635
...,...,...,...,...,...,...
371,V317,float64,590528,12,0.0020,15184
372,V318,float64,590528,12,0.0020,12309
373,V319,float64,590528,12,0.0020,4799
374,V320,float64,590528,12,0.0020,6439


['dist2' 'D7' 'D13' 'D14' 'D12' 'D6' 'D8' 'D9' 'V138' 'V139' 'V140' 'V141'
 'V142' 'V146' 'V147' 'V148' 'V149' 'V153' 'V154' 'V155' 'V156' 'V157'
 'V158' 'V161' 'V162' 'V163' 'V143' 'V144' 'V145' 'V150' 'V151' 'V152'
 'V159' 'V160' 'V164' 'V165' 'V166' 'V322' 'V323' 'V324' 'V325' 'V326'
 'V327' 'V328' 'V329' 'V330' 'V331' 'V332' 'V333' 'V334' 'V335' 'V336'
 'V337' 'V338' 'V339' 'V217' 'V218' 'V219' 'V223' 'V224' 'V225' 'V226'
 'V228' 'V229' 'V230' 'V231' 'V232' 'V233' 'V235' 'V236' 'V237' 'V240'
 'V241' 'V242' 'V243' 'V244' 'V246' 'V247' 'V248' 'V249' 'V252' 'V253'
 'V254' 'V257' 'V258' 'V260' 'V261' 'V262' 'V263' 'V264' 'V265' 'V266'
 'V267' 'V268' 'V269' 'V273' 'V274' 'V275' 'V276' 'V277' 'V278'
 'R_emaildomain' 'V167' 'V168' 'V172' 'V173' 'V176' 'V177' 'V178' 'V179'
 'V181' 'V182' 'V183' 'V186' 'V187' 'V190' 'V191' 'V192' 'V193' 'V196'
 'V199' 'V202' 'V203' 'V204' 'V205' 'V206' 'V207' 'V211' 'V212' 'V213'
 'V214' 'V215' 'V216' 'V169' 'V170' 'V171' 'V174' 'V175' 'V180' 'V184'
 'V185'

In [13]:
# TransactionDT is elapsed time from an undisclosed reference point, not a calendar timestamp.
if "TransactionDT" in ieee_transaction.columns:
    dt = ieee_transaction["TransactionDT"]
    print(f"TransactionDT minimum: {dt.min():,.0f}")
    print(f"TransactionDT maximum: {dt.max():,.0f}")
    print(f"Approximate duration: {(dt.max() - dt.min()) / 86400:,.2f} days")
    ieee_transaction["audit_day"] = (dt // 86400).astype("Int64")
    display(ieee_transaction.groupby("audit_day").size().rename("transactions").head(10).to_frame())
else:
    print("TransactionDT not found; relative time summary skipped.")

TransactionDT minimum: 86,400
TransactionDT maximum: 15,811,131
Approximate duration: 182.00 days


,transactions
audit_day,
1,5122
2,3730
3,3241
4,4036
5,3964
6,3717
7,3786
8,3999
9,3907


In [14]:
# Optional IEEE-CIS identity table audit.
if LOAD_OPTIONAL_FILES and Path(IEEE_IDENTITY_PATH).exists():
    ieee_identity = read_csv_checked(IEEE_IDENTITY_PATH)
    display(ieee_identity.head())
    ieee_identity_schema = schema_audit(ieee_identity, "IEEE-CIS train identity")
    if "TransactionID" in ieee_transaction.columns and "TransactionID" in ieee_identity.columns:
        transaction_ids = set(ieee_transaction["TransactionID"].dropna())
        identity_ids = set(ieee_identity["TransactionID"].dropna())
        print(f"Transactions with identity records: {len(transaction_ids & identity_ids):,} / {len(transaction_ids):,} ({len(transaction_ids & identity_ids) / len(transaction_ids):.2%})")
        print(f"Identity rows without a matching transaction: {len(identity_ids - transaction_ids):,}")
else:
    print("Optional IEEE-CIS identity file not found or disabled.")

Reading ..\data\raw\train_identity.csv ...


,TransactionID,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,id_10,id_11,id_12,id_13,id_14,id_15,id_16,id_17,id_18,id_19,id_20,id_21,id_22,id_23,id_24,id_25,id_26,id_27,id_28,id_29,id_30,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987004,0.0000,"70,787.0000",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0000,NotFound,NaN,-480.0000,New,NotFound,166.0000,NaN,542.0000,144.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,New,NotFound,Android 7.0,samsung browser 6.2,32.0000,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M
1,2987008,-5.0000,"98,945.0000",NaN,NaN,0.0000,-5.0000,NaN,NaN,NaN,NaN,100.0000,NotFound,49.0000,-300.0000,New,NotFound,166.0000,NaN,621.0000,500.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,New,NotFound,iOS 11.1.2,mobile safari 11.0,32.0000,1334x750,match_status:1,T,F,F,T,mobile,iOS Device
2,2987010,-5.0000,"191,631.0000",0.0000,0.0000,0.0000,0.0000,NaN,NaN,0.0000,0.0000,100.0000,NotFound,52.0000,NaN,Found,Found,121.0000,NaN,410.0000,142.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Found,Found,NaN,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,Windows
3,2987011,-5.0000,"221,832.0000",NaN,NaN,0.0000,-6.0000,NaN,NaN,NaN,NaN,100.0000,NotFound,52.0000,NaN,New,NotFound,225.0000,NaN,176.0000,507.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,New,NotFound,NaN,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,NaN
4,2987016,0.0000,"7,460.0000",0.0000,0.0000,1.0000,0.0000,NaN,NaN,0.0000,0.0000,100.0000,NotFound,NaN,-300.0000,Found,Found,166.0000,15.0000,529.0000,575.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Found,Found,Mac OS X 10_11_6,chrome 62.0,24.0000,1280x800,match_status:2,T,F,T,T,desktop,MacOS


IEEE-CIS train identity: 144,233 rows × 41 columns
Duplicate rows: 0


,column,dtype,non_null,missing,missing_pct,n_unique
0,id_24,float64,4747,139486,96.7088,12
1,id_25,float64,5132,139101,96.4419,341
2,id_07,float64,5155,139078,96.4259,84
3,id_08,float64,5155,139078,96.4259,94
4,id_21,float64,5159,139074,96.4231,490
5,id_26,float64,5163,139070,96.4204,95
6,id_22,float64,5169,139064,96.4162,25
7,id_23,object,5169,139064,96.4162,3
8,id_27,object,5169,139064,96.4162,2
9,id_18,float64,45113,99120,68.7221,18


Transactions with identity records: 144,233 / 590,540 (24.42%)
Identity rows without a matching transaction: 0


# Compact Summary

In [15]:
summary_rows = []
for name, df, label in [
    ("Sparkov train", sparkov_train, "is_fraud"),
    ("IEEE-CIS train transaction", ieee_transaction, "isFraud"),
]:
    labelled = df[label].notna().sum() if label in df else np.nan
    fraud = (df[label] == 1).sum() if label in df else np.nan
    summary_rows.append({
        "dataset": name,
        "rows": len(df),
        "columns": df.shape[1],
        "duplicate_rows": df.duplicated().sum(),
        "labelled_rows": labelled,
        "fraud_rows": fraud,
        "fraud_rate_among_labelled_pct": fraud / labelled * 100 if labelled else np.nan,
        "columns_with_missing_values": (df.isna().any()).sum(),
    })
audit_summary = pd.DataFrame(summary_rows)
display(audit_summary)

,dataset,rows,columns,duplicate_rows,labelled_rows,fraud_rows,fraud_rate_among_labelled_pct,columns_with_missing_values
0,Sparkov train,1296675,24,0,1296675,7506,0.5789,0
1,IEEE-CIS train transaction,590540,395,0,590540,20663,3.4990,374
